## Descrição do Processo do Notebook

Este notebook executa o processo de **preparação e ingestão de dados** para a base de dados de contratos públicos.

---

### Fluxo de Processamento de Dados

Abaixo está o detalhe de cada etapa executada pelo notebook, com a seta $(\rightarrow)$ a indicar o próximo passo no fluxo de trabalho.

| Passo | Descrição |
| :---: | :--- |
| **1º Step** |  **Input:** Leitura do ficheiro **`ContratosPublicos2024.csv`**. $\rightarrow$ |
| **2º Step** |  **Colunas para CSV:** As colunas do ficheiro original são extraídas e utilizadas para gerar **CSVs individuais** para cada conjunto de dados ou categoria relevante. $\rightarrow$ |
| **3º Step** |  **Tratar CSV Individualmente:** Cada CSV individual gerado no passo anterior é processado e **tratado** (limpeza, normalização, transformação, etc.) de forma autónoma. $\rightarrow$ |
| **4º Step** |  **Map to new `ContratosPublicos2024.csv`:** Os dados tratados são mapeados e consolidados num **novo** ficheiro `ContratosPublicos2024.csv`. $\rightarrow$ |
| **5º Step** | **Create Database `ContratosPublicos2024.db`:** O ficheiro CSV final é usado para **criar a base de dados** `ContratosPublicos2024.db` e preenchê-la com os dados processados. |

In [20]:
import pandas as pd
import os
os.makedirs('tabelas2', exist_ok=True)
diretorio_destino = 'tabelas2'
df = pd.read_excel('ContratosPublicos2024.xlsx')
df.head()
dataframes_por_coluna = {}
for nome_coluna in df.columns:
    novo_df = df[[nome_coluna]].copy()
    nome_arquivo_limpo = nome_coluna.replace(' ', '_').replace('/', '_')
    caminho_completo = os.path.join(diretorio_destino, f'{nome_arquivo_limpo}.csv')
    novo_df.to_csv(caminho_completo, index=False)

In [21]:
import pandas as pd
import os

# --- Configurações ---
ARQUIVO_ADJUDICANTES = 'tabelas2/adjudicante.csv'
ARQUIVO_ADJUDICATARIOS = 'tabelas2/adjudicatarios.csv'
ARQUIVO_SAIDA = 'tabelas2/entidades.csv'

# Nomes das colunas combinadas em CADA arquivo (Ajuste se estes nomes não estiverem corretos!)
COLUNA_COMBINADA_ADJUDICANTE = 'adjudicante'
COLUNA_COMBINADA_ADJUDICATARIO = 'adjudicatarios'

COLUNAS_PARA_UNICIDADE = ['NIF', 'Nome'] 
NOME_COLUNA_UNIFICADA = 'Entidade_Completa' # Nome temporário para a coluna combinada após o carregamento

# --- 1. Carregar, Selecionar e Renomear as Colunas ---
try:
    # 1.1 Carregar APENAS a coluna relevante de cada arquivo
    df_adjudicantes = pd.read_csv(ARQUIVO_ADJUDICANTES, 
                                  usecols=[COLUNA_COMBINADA_ADJUDICANTE])
    
    df_adjudicatarios = pd.read_csv(ARQUIVO_ADJUDICATARIOS, 
                                    usecols=[COLUNA_COMBINADA_ADJUDICATARIO])
    
    # 1.2 Renomear a coluna em cada DataFrame para um nome UNIFICADO
    df_adjudicantes.rename(columns={COLUNA_COMBINADA_ADJUDICANTE: NOME_COLUNA_UNIFICADA}, 
                           inplace=True)
    df_adjudicatarios.rename(columns={COLUNA_COMBINADA_ADJUDICATARIO: NOME_COLUNA_UNIFICADA}, 
                             inplace=True)
    
    print(f"Lido '{ARQUIVO_ADJUDICANTES}' ({len(df_adjudicantes)} linhas) e coluna renomeada para '{NOME_COLUNA_UNIFICADA}'.")
    print(f"Lido '{ARQUIVO_ADJUDICATARIOS}' ({len(df_adjudicatarios)} linhas) e coluna renomeada para '{NOME_COLUNA_UNIFICADA}'.")

except FileNotFoundError as e:
    print(f"ERRO: Não foi possível encontrar o arquivo: {e.filename}. Certifique-se de que os arquivos estão no mesmo diretório do script.")
    exit()
except ValueError:
    print(f"ERRO: Verifique os nomes das colunas. As colunas de entrada ('{COLUNA_COMBINADA_ADJUDICANTE}' e '{COLUNA_COMBINADA_ADJUDICATARIO}') não foram encontradas.")
    exit()

# --- 2. Divisão da Coluna Unificada ---

# Processar ambos os DataFrames (agora ambos têm a coluna NOME_COLUNA_UNIFICADA)
for df in [df_adjudicantes, df_adjudicatarios]:
    
    # Padronizar a coluna combinada (remover espaços extras e preencher vazios)
    df[NOME_COLUNA_UNIFICADA] = df[NOME_COLUNA_UNIFICADA].fillna('').astype(str).str.strip()
    
    # Usar .str.split() para dividir a coluna pelo primeiro ' - ' encontrado
    # n=1 garante que divide apenas na primeira ocorrência: "505111667" e "Urbe - Consultores..."
    df[['NIF', 'Nome']] = df[NOME_COLUNA_UNIFICADA].str.split(' - ', n=1, expand=True)

    # Limpar espaços em branco adicionais que possam surgir após a divisão
    df['NIF'] = df['NIF'].str.strip()
    df['Nome'] = df['Nome'].str.strip()
    
    # Remover a coluna combinada original
    df.drop(columns=[NOME_COLUNA_UNIFICADA], inplace=True)


# --- 3. Concatenação e Limpeza ---

# Concatenar (empilhar) os DataFrames
df_entidades_com_duplicatas = pd.concat(
    [df_adjudicantes, df_adjudicatarios], 
    ignore_index=True
)

linhas_totais = len(df_entidades_com_duplicatas)
print(f"\nTotal de linhas combinadas (com duplicatas): {linhas_totais}")

# Tratar casos onde NIF/Nome pode ser NaN após a divisão
df_entidades_com_duplicatas['NIF'] = df_entidades_com_duplicatas['NIF'].fillna('')
df_entidades_com_duplicatas['Nome'] = df_entidades_com_duplicatas['Nome'].fillna('')


# --- 4. Garantir Unicidade (Remover Duplicatas) ---

# Remover duplicatas com base no par (NIF, Nome)
df_entidades_unicas = df_entidades_com_duplicatas.drop_duplicates(
    subset=COLUNAS_PARA_UNICIDADE, 
    keep='first'
)

linhas_unicas_sem_id = len(df_entidades_unicas)
duplicatas_removidas = linhas_totais - linhas_unicas_sem_id

print(f"Duplicatas removidas (baseado em NIF e Nome): {duplicatas_removidas}")

# --- 5. Gerar Coluna ID Único e Reordenar ---

df_entidades_unicas = df_entidades_unicas.reset_index(drop=True)
df_entidades_unicas['id'] = df_entidades_unicas.index + 1

# Selecionar e reordenar as colunas finais (id, NIF, Nome)
df_entidades_final = df_entidades_unicas[['id', 'NIF', 'Nome']]

# --- 6. Guardar o CSV de Saída ---
df_entidades_final.to_csv(ARQUIVO_SAIDA, index=False)

# --- Fim ---
print(f"\n✨ SUCESSO! O arquivo final '{ARQUIVO_SAIDA}' foi criado com {len(df_entidades_final)} linhas únicas.")
    # Us

Lido 'tabelas2/adjudicante.csv' (21748 linhas) e coluna renomeada para 'Entidade_Completa'.
Lido 'tabelas2/adjudicatarios.csv' (21748 linhas) e coluna renomeada para 'Entidade_Completa'.

Total de linhas combinadas (com duplicatas): 43496
Duplicatas removidas (baseado em NIF e Nome): 32091

✨ SUCESSO! O arquivo final 'tabelas2/entidades.csv' foi criado com 11405 linhas únicas.


In [22]:
import pandas as pd
import os

# Lista dos caminhos completos dos ficheiros a processar
FICHEIROS_A_PROCESSAR = [
    'tabelas2/tipoprocedimento.csv',
    'tabelas2/fundamentacao.csv',
    'tabelas2/DescrAcordoQuadro.csv',
    'tabelas2/localExecucao.csv',
    'tabelas2/tipoContrato.csv'
]

# Itera sobre cada ficheiro na lista
for caminho_completo in FICHEIROS_A_PROCESSAR:
    
    # Extrai apenas o nome do ficheiro para mensagens
    nome_ficheiro = os.path.basename(caminho_completo)
    
    print(f"\n--- Processando: {nome_ficheiro} ---")

    try:
        # 1. Carregar o DataFrame
        df = pd.read_csv(caminho_completo)
        linhas_originais = len(df)
        print(f"Linhas originais: {linhas_originais}")
        
    except FileNotFoundError:
        print(f"⚠️ ERRO: Ficheiro não encontrado no caminho: {caminho_completo}")
        continue # Passa para o próximo ficheiro na lista
    
    # 2. Remover Linhas com Nulos (NULLs/NaNs)
    
    # Antes da remoção
    nulos_antes = df.isnull().any(axis=1).sum()
    
    # Remove qualquer linha onde pelo menos um valor seja nulo (NaN)
    df.dropna(how='any', inplace=True)
    
    nulos_removidos = nulos_antes
    
    print(f"Linhas com nulos removidas: {nulos_removidos}")
    
    # 3. Remover Repetidos (Duplicatas)
    
    # Antes da remoção
    duplicatas_antes = df.duplicated().sum()
    
    # Remove linhas duplicadas (mantém a primeira ocorrência)
    df.drop_duplicates(keep='first', inplace=True)
    
    duplicatas_removidas = duplicatas_antes
    
    print(f"Linhas duplicadas removidas: {duplicatas_removidas}")
    
    # 4. Criar a Coluna ID Único
    
    # Resetar o índice para garantir uma sequência limpa (de 0 a N-1)
    df = df.reset_index(drop=True)
    
    # Criar a coluna 'ID' começando em 1
    df['ID'] = df.index + 1
    
    # 5. Reordenar as colunas para colocar o ID no início
    
    colunas_novas = ['ID'] + [col for col in df.columns if col != 'ID']
    df = df[colunas_novas]
    
    # 6. Salvar o DataFrame processado (sobrescrevendo o original, ou use um novo nome)
    df.to_csv(caminho_completo, index=False)
    
    print(f"Linhas finais: {len(df)}")
    print(f"✅ Ficheiro '{nome_ficheiro}' processado e salvo com a coluna 'ID'.")

print("\n--- Processamento de todos os ficheiros concluído. ---")


--- Processando: tipoprocedimento.csv ---
Linhas originais: 21748
Linhas com nulos removidas: 0
Linhas duplicadas removidas: 21737
Linhas finais: 11
✅ Ficheiro 'tipoprocedimento.csv' processado e salvo com a coluna 'ID'.

--- Processando: fundamentacao.csv ---
Linhas originais: 21748
Linhas com nulos removidas: 137
Linhas duplicadas removidas: 21520
Linhas finais: 91
✅ Ficheiro 'fundamentacao.csv' processado e salvo com a coluna 'ID'.

--- Processando: DescrAcordoQuadro.csv ---
Linhas originais: 21748
Linhas com nulos removidas: 18203
Linhas duplicadas removidas: 3289
Linhas finais: 256
✅ Ficheiro 'DescrAcordoQuadro.csv' processado e salvo com a coluna 'ID'.

--- Processando: localExecucao.csv ---
Linhas originais: 21748
Linhas com nulos removidas: 0
Linhas duplicadas removidas: 21138
Linhas finais: 610
✅ Ficheiro 'localExecucao.csv' processado e salvo com a coluna 'ID'.

--- Processando: tipoContrato.csv ---
Linhas originais: 21748
Linhas com nulos removidas: 0
Linhas duplicadas remo

In [23]:
import pandas as pd
import os

# --- Configurações ---
CAMINHO_FICHEIRO = 'tabelas2/localExecucao.csv'
COLUNA_COMBINADA = 'localExecucao'
NOVAS_COLUNAS = ['pais', 'distrito', 'concelho']

print(f"--- Processando: {os.path.basename(CAMINHO_FICHEIRO)} ---")

try:
    # 1. Carregar o DataFrame
    df = pd.read_csv(CAMINHO_FICHEIRO)
    linhas_originais = len(df)
    print(f"Linhas originais: {linhas_originais}")
    
    if COLUNA_COMBINADA not in df.columns:
        print(f"⚠️ ERRO: A coluna '{COLUNA_COMBINADA}' não foi encontrada no ficheiro. Verifique a escrita.")
        exit()
        
except FileNotFoundError:
    print(f"⚠️ ERRO: Ficheiro não encontrado no caminho: {CAMINHO_FICHEIRO}")
    exit()

# 2. Padronizar a coluna combinada
# Tratar como string e remover espaços antes de dividir
df[COLUNA_COMBINADA] = df[COLUNA_COMBINADA].astype(str).str.strip()

# 3. Dividir a coluna 'localExecucao' (O FIX ESTÁ AQUI!)
# Usamos n=2 para garantir que a divisão ocorra no máximo 2 vezes,
# resultando sempre em 3 colunas (país, distrito, concelho), mesmo que haja mais vírgulas no dado.
df[NOVAS_COLUNAS] = df[COLUNA_COMBINADA].str.split(',', n=2, expand=True)

# 4. Limpeza Pós-Divisão e Remoção da Coluna Original

# Aplicar .str.strip() para remover espaços em branco nas novas colunas
for col in NOVAS_COLUNAS:
    # Tratar valores que possam ter ficado vazios ou NaN
    df[col] = df[col].astype(str).str.strip().replace('nan', '') 
    
# Remover a Coluna Combinada Original
df.drop(columns=[COLUNA_COMBINADA], inplace=True)

# 5. Salvar o DataFrame Atualizado
df.to_csv(CAMINHO_FICHEIRO, index=False)

print(f"\nAs colunas {NOVAS_COLUNAS} foram criadas.")
print(f"Coluna '{COLUNA_COMBINADA}' original foi removida.")
print(f"Linhas finais: {len(df)}")
print(f"✅ Ficheiro '{os.path.basename(CAMINHO_FICHEIRO)}' salvo com as novas colunas de localização.")

--- Processando: localExecucao.csv ---
Linhas originais: 610

As colunas ['pais', 'distrito', 'concelho'] foram criadas.
Coluna 'localExecucao' original foi removida.
Linhas finais: 610
✅ Ficheiro 'localExecucao.csv' salvo com as novas colunas de localização.


In [24]:
import pandas as pd
import os
import re

# --- Configurações ---
CAMINHO_ORIGINAL = 'ContratosPublicos2024.xlsx'
CAMINHO_FICHEIRO = 'tabelas2/fundamentacao.csv'
COLUNA_COMBINADA = 'fundamentacao'

print(f"--- Processando: {os.path.basename(CAMINHO_FICHEIRO)} ---")

try:
    # 1. Carregar o DataFrame
    df = pd.read_excel(CAMINHO_ORIGINAL)
    out = pd.read_csv(CAMINHO_FICHEIRO)
    linhas_originais = len(df)
    print(f"Linhas originais: {linhas_originais}")
    
    if COLUNA_COMBINADA not in df.columns:
        print(f"⚠️ ERRO: A coluna '{COLUNA_COMBINADA}' não foi encontrada no ficheiro. Verifique a escrita.")
        exit()
        
except FileNotFoundError:
    print(f"⚠️ ERRO: Ficheiro não encontrado no caminho: {CAMINHO_FICHEIRO}")
    exit()

# 2. Padronizar a coluna combinada e extrair dados
df[COLUNA_COMBINADA] = df[COLUNA_COMBINADA].astype(str).str.strip()

# --- Expressões Regulares ---
# Captura o número do Artigo, independentemente do que vier antes ou depois.
REGEX_ARTIGO = r'Artigo\s*(\d+\.º|\d+)'

# Captura o número do parágrafo, se existir (n.º X)
REGEX_NUMERO = r'n\.º\s*(\d+)'

# Captura a alínea, se existir (alínea X)
REGEX_ALINEA = r'alínea\s*([a-z])'

# 3. Criar as Novas Colunas usando .str.extract()
# Use a flag re.IGNORECASE para ignorar maiúsculas/minúsculas ("Artigo" vs "ARTIGO")
# Use a flag re.IGNORECASE para ignorar maiúsculas/minúsculas ("Artigo" vs "ARTIGO")
out['Artigo'] = df[COLUNA_COMBINADA].str.extract(REGEX_ARTIGO, flags=re.IGNORECASE)

# Extrai o Número (n.º):
out['Numero_n'] = df[COLUNA_COMBINADA].str.extract(REGEX_NUMERO, flags=re.IGNORECASE)

# Extrai a Alínea:
out['Alinea'] = df[COLUNA_COMBINADA].str.extract(REGEX_ALINEA, flags=re.IGNORECASE)
#Trata
out['Artigo'] = out['Artigo'].str.replace('\.º', '', regex=True).fillna('').str.strip().astype(str)
out['Numero_n'] = out['Numero_n'].fillna('').str.strip().astype(str)
out['Alinea'] = out['Alinea'].fillna('').str.strip().astype(str)
# 5. Remover a Coluna Combinada Original
df.drop(columns=[COLUNA_COMBINADA], inplace=True)
# 6. Salvar o DataFrame Atualizado
out.to_csv(CAMINHO_FICHEIRO, index=False)

print(f"\nAs colunas 'Artigo', 'Numero_n' e 'Alinea' foram criadas.")
print(f"Coluna '{COLUNA_COMBINADA}' original foi removida.")
print(f"Linhas finais: {len(df)}")
print(f"✅ Ficheiro '{os.path.basename(CAMINHO_FICHEIRO)}' salvo com as novas colunas de fundamentação legal.")

--- Processando: fundamentacao.csv ---


<>:51: SyntaxWarning: invalid escape sequence '\.'
<>:51: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_226790/2875771222.py:51: SyntaxWarning: invalid escape sequence '\.'
  out['Artigo'] = out['Artigo'].str.replace('\.º', '', regex=True).fillna('').str.strip().astype(str)


Linhas originais: 21748

As colunas 'Artigo', 'Numero_n' e 'Alinea' foram criadas.
Coluna 'fundamentacao' original foi removida.
Linhas finais: 21748
✅ Ficheiro 'fundamentacao.csv' salvo com as novas colunas de fundamentação legal.


In [25]:
import pandas as pd
import os

# --- Configurações ---
# Substitua 'cpv.csv' pelo caminho completo se não estiver no diretório de trabalho
CAMINHO_FICHEIRO = 'tabelas2/cpv.csv' 
COLUNA_COMBINADA = 'cpv'
NOVAS_COLUNAS = ['Codigo_CPV', 'Descricao']
DELIMITADOR = ' - ' # Hífen com espaços ao lado

print(f"--- Processando: {os.path.basename(CAMINHO_FICHEIRO)} ---")

try:
    # 1. Carregar o DataFrame
    df = pd.read_csv(CAMINHO_FICHEIRO)
    linhas_originais = len(df)
    print(f"Linhas originais: {linhas_originais}")
    
    if COLUNA_COMBINADA not in df.columns:
        print(f"⚠️ ERRO: A coluna '{COLUNA_COMBINADA}' não foi encontrada no ficheiro. Verifique a escrita.")
        exit()
        
except FileNotFoundError:
    print(f"⚠️ ERRO: Ficheiro não encontrado no caminho: {CAMINHO_FICHEIRO}")
    exit()

# 2. Padronizar a coluna combinada
# Tratar como string e remover espaços antes de dividir
df[COLUNA_COMBINADA] = df[COLUNA_COMBINADA].astype(str).str.strip()

# 3. Dividir a coluna 'cpv'
# Usamos n=1 para garantir que a divisão ocorra apenas na primeira ocorrência de ' - ',
# resultando em 2 colunas: Código e Descrição.
df[NOVAS_COLUNAS] = df[COLUNA_COMBINADA].str.split(DELIMITADOR, n=1, expand=True)

# 4. Remover Duplicatas (Com base no código e na descrição)
# Os seus dados têm exemplos de duplicatas que devem ser removidas (90611000-3).
df_antes_duplicatas = len(df)
df.drop_duplicates(subset=NOVAS_COLUNAS, keep='first', inplace=True)
duplicatas_removidas = df_antes_duplicatas - len(df)
print(f"Linhas duplicadas removidas: {duplicatas_removidas}")


# 5. Limpeza Pós-Divisão
for col in NOVAS_COLUNAS:
    # Aplicar .str.strip() para remover espaços e tratar NaNs como string vazia
    df[col] = df[col].astype(str).str.strip().replace('nan', '') 
    
# 6. Adicionar a coluna ID (se ela não existir, assumindo que foi o passo anterior)
if 'ID' not in df.columns:
    df = df.reset_index(drop=True)
    df['ID'] = df.index + 1
    colunas_finais = ['ID'] + NOVAS_COLUNAS
    df = df[colunas_finais]


# 7. Remover a Coluna Combinada Original
#df.drop(columns=[COLUNA_COMBINADA], inplace=True)

# 8. Salvar o DataFrame Atualizado
df.to_csv(CAMINHO_FICHEIRO, index=False)

print(f"\nAs colunas {NOVAS_COLUNAS} foram criadas.")
print(f"Coluna '{COLUNA_COMBINADA}' original foi removida.")
print(f"Linhas finais: {len(df)}")
print(f"✅ Ficheiro '{os.path.basename(CAMINHO_FICHEIRO)}' salvo com as novas colunas de CPV.")

--- Processando: cpv.csv ---
Linhas originais: 21748
Linhas duplicadas removidas: 19480

As colunas ['Codigo_CPV', 'Descricao'] foram criadas.
Coluna 'cpv' original foi removida.
Linhas finais: 2268
✅ Ficheiro 'cpv.csv' salvo com as novas colunas de CPV.


In [26]:
"""
Criacao contratos

"""
import pandas as pd

nome_arquivo = 'contratos.csv'

df1 = pd.read_csv("tabelas2/idcontrato.csv")
df2 = pd.read_csv("tabelas2/prazoExecucao.csv")
df3 = pd.read_csv("tabelas2/precoContratual.csv")
df4 = pd.read_csv("tabelas2/dataCelebracaoContrato.csv")
df5 = pd.read_csv("tabelas2/dataPublicacao.csv")
df6 = pd.read_csv("tabelas2/ProcedimentoCentralizado.csv")
df7 = pd.read_csv("tabelas2/objectoContrato.csv")

# 1. Corrija o nome da coluna chave em todos os DataFrames (VITAL)
# Se ainda tiver dúvidas sobre o KeyError, execute este bloco:
dataframes_para_concatenar = [df1, df2, df3, df4, df5, df6, df7]

# O argumento 'axis=1' garante que os DataFrames são unidos horizontalmente
contratos = pd.concat(dataframes_para_concatenar, axis=1)
print("\n🎉 DataFrame 'contratos' unido corretamente!")
contratos.to_csv('tabelas2/contratos.csv',index=False,encoding='utf-8')
print(contratos.head())


🎉 DataFrame 'contratos' unido corretamente!
   idcontrato  prazoExecucao  precoContratual dataCelebracaoContrato  \
0    10424261            366          3918.75             2024-01-01   
1    10424271            366           467.95             2024-01-02   
2    10424433             30         27849.00             2024-01-02   
3    10424474            365         11520.00             2024-01-02   
4    10424593            365         19248.00             2024-01-02   

  dataPublicacao ProcedimentoCentralizado  \
0     2024-01-01                      Não   
1     2024-01-02                      Não   
2     2024-01-02                      Não   
3     2024-01-02                      Não   
4     2024-01-02                      Não   

                                     objectoContrato  
0  Seguro de Acidentes de Trabalho para os Funcio...  
1          Seguro Automóvel - FIAT 500L/1.3MJ LOUNGE  
2              Ligação EN 342 a placa de S. Martinho  
3  Serviços de limpeza de berma

In [27]:
import pandas as pd
contratos_original = pd.read_excel("ContratosPublicos2024.xlsx")
contratos = pd.read_csv('tabelas2/contratos.csv')
contratos_original['Codigo_CPV'] = contratos_original['cpv'].str.split(' - ').str[0]
id_contratos = pd.read_csv('tabelas2/cpv.csv')
contratos_merged = pd.merge(
    contratos_original,
    id_contratos,
    on='Codigo_CPV', 
    how='left'         
)
contratos['cpv'] = contratos_merged['ID']
contratos['cpv'].head()
contratos.to_csv('tabelas2/contratos.csv')



In [28]:
import pandas as pd
contratos_original = pd.read_excel("ContratosPublicos2024.xlsx")
contratos = pd.read_csv('tabelas2/contratos.csv')
id_contratos = pd.read_csv('tabelas2/tipoContrato.csv')

contratos_merged = pd.merge(
    contratos_original,
    id_contratos,
    on='tipoContrato', 
    how='left'         
)
contratos['tipoContrato'] = contratos_merged['ID']
contratos.to_csv('tabelas2/contratos.csv')


In [29]:
import pandas as pd
contratos_original = pd.read_excel("ContratosPublicos2024.xlsx")
contratos = pd.read_csv('tabelas2/contratos.csv')
id_contratos = pd.read_csv('tabelas2/tipoprocedimento.csv')
contratos_merged = pd.merge(
    contratos_original,
    id_contratos,
    on='tipoprocedimento', 
    how='left'         
)
contratos['tipoprocedimento'] = contratos_merged['ID']
contratos.head()

contratos.to_csv('tabelas2/contratos.csv')

In [30]:
import pandas as pd
contratos_original = pd.read_excel("ContratosPublicos2024.xlsx")
contratos = pd.read_csv('tabelas2/contratos.csv')
id_contratos = pd.read_csv('tabelas2/entidades.csv')
contratos_original['NIF']=contratos_original['adjudicante'].str.split(' - ',expand=True)[0].str.strip()
contratos_merged = pd.merge(
    contratos_original,
    id_contratos,
    on='NIF', 
    how='left'         
)
contratos['adjudicante'] = contratos_merged['id']
contratos.head()

contratos.to_csv('tabelas2/contratos.csv')

In [31]:
import pandas as pd
contratos_original = pd.read_excel("ContratosPublicos2024.xlsx")
contratos = pd.read_csv('tabelas2/contratos.csv')
id_contratos = pd.read_csv('tabelas2/entidades.csv')
contratos_original['NIF']=contratos_original['adjudicatarios'].str.split(' - ',expand=True)[0].str.strip()
contratos_merged = pd.merge(
    contratos_original,
    id_contratos,
    on='NIF', 
    how='left'         
)
contratos['adjudicatarios'] = contratos_merged['id']
contratos.head()

contratos.to_csv('tabelas2/contratos.csv')

In [33]:
import pandas as pd
import re # Necessário para usar a flag re.IGNORECASE no .str.extract()
contratos = pd.read_csv('tabelas2/contratos.csv')

# Supondo que a coluna a limpar no contratos_original se chame 'Clausula_Legal'
COLUNA_COMBINADA = 'fundamentacao' 
# --- A sua lógica de carregamento de dados ---
contratos_original = pd.read_excel("ContratosPublicos2024.xlsx")
df = contratos_original # Usar df apenas para seguir o seu exemplo

# 2. Padronizar a coluna combinada e extrair dados
df[COLUNA_COMBINADA] = df[COLUNA_COMBINADA].astype(str).str.strip()

# --- Expressões Regulares ---
# Captura o número do Artigo, independentemente do que vier antes ou depois.
REGEX_ARTIGO = r'Artigo\s*(\d+\.º|\d+)'

# Captura o número do parágrafo, se existir (n.º X)
REGEX_NUMERO = r'n\.º\s*(\d+)'

# Captura a alínea, se existir (alínea X)
REGEX_ALINEA = r'alínea\s*([a-z])'


# 3. Criar as Novas Colunas usando .str.extract()
# Use a flag re.IGNORECASE para ignorar maiúsculas/minúsculas ("Artigo" vs "ARTIGO")
df['Artigo'] = df[COLUNA_COMBINADA].str.extract(REGEX_ARTIGO, flags=re.IGNORECASE)

# Extrai o Número (n.º):
df['Numero_n'] = df[COLUNA_COMBINADA].str.extract(REGEX_NUMERO, flags=re.IGNORECASE)

# Extrai a Alínea:
df['Alinea'] = df[COLUNA_COMBINADA].str.extract(REGEX_ALINEA, flags=re.IGNORECASE)


# 4. Limpeza Pós-Extração (Limpar o Artigo e preencher NaN's)
# Limpar 'Artigo' para remover o '.º' (o regex=True é crucial aqui)
df['Artigo'] = df['Artigo'].str.replace('\.º', '', regex=True).fillna('').str.strip().astype(str)
df['Numero_n'] = df['Numero_n'].fillna('').str.strip().astype(str)
df['Alinea'] = df['Alinea'].fillna('').str.strip().astype(str)

id_contratos = pd.read_csv('tabelas2/fundamentacao.csv')
id_contratos['Artigo'] = id_contratos['Artigo'].astype(str).str.strip()
id_contratos['Numero_n'] = id_contratos['Numero_n'].astype(str).str.strip()
id_contratos['Alinea'] = id_contratos['Alinea'].astype(str).str.strip()
for col in ['Artigo', 'Numero_n']:
    # Se o CSV carregou como float (ex: 20.0), a conversão para str será '20.0'.
    # Usamos regex para limpar o .0 (no caso de Artigo/Numero)
    id_contratos[col] = (id_contratos[col].astype(str)
                                           .str.replace(r'(\.º|\.0)', '', regex=True) # Remove .0 ou .º
                                           .str.replace('nan', '', regex=False)       # Substitui "nan" por string vazia ""
                                           .str.strip())

# 🎯 Passo Crucial 2: Limpeza da Alinea e conversão de 'nan' para ''
id_contratos['Alinea'] = (id_contratos['Alinea'].astype(str)
                                                .str.lower()
                                                .str.replace('nan', '', regex=False) # Substitui "nan" por string vazia ""
                                                .str.strip())
id_contratos['ID'] = (id_contratos['ID'].astype(str)
                                                .str.lower()
                                                .str.replace('nan', '', regex=False) # Substitui "nan" por string vazia ""
                                                .str.strip())
print(df[['Artigo', 'Numero_n', 'Alinea']].head())
print(id_contratos[['Artigo', 'Numero_n', 'Alinea']].head())

contratos_merged = pd.merge(
    df,
    id_contratos[['Artigo','Numero_n','Alinea','ID']],
    on=['Artigo','Numero_n','Alinea'],
    how='left'         
)
contratos['fundamentacao'] = contratos_merged['ID']
contratos.to_csv('tabelas2/contratos.csv')
print(contratos['fundamentacao'].head())

<>:38: SyntaxWarning: invalid escape sequence '\.'
<>:38: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_226790/28281336.py:38: SyntaxWarning: invalid escape sequence '\.'
  df['Artigo'] = df['Artigo'].str.replace('\.º', '', regex=True).fillna('').str.strip().astype(str)


  Artigo Numero_n Alinea
0     20        1      c
1     20        1      c
2     19               c
3     20        1      d
4     20        1      c
  Artigo Numero_n Alinea
0     20        1      c
1     20        1      c
2     19               c
3     20        1      d
4     20        1      c
0    1
1    2
2    5
3    8
4    9
Name: fundamentacao, dtype: object


In [34]:
import pandas as pd
contratos = pd.read_csv('tabelas2/contratos.csv')
contratos_original = pd.read_excel('ContratosPublicos2024.xlsx')
id_contratos = pd.read_csv('tabelas2/DescrAcordoQuadro.csv')
contratos_merged = pd.merge(
    contratos_original,
    id_contratos,
    on='DescrAcordoQuadro', 
    how='left'         
)
contratos['DescrAcordoQuadro'] = contratos_merged['ID']
contratos['DescrAcordoQuadro'].head(25)
contratos.to_csv('tabelas2/contratos.csv')

In [35]:
import pandas as pd
contratos = pd.read_csv('tabelas2/contratos.csv')
contratos_original = pd.read_excel('ContratosPublicos2024.xlsx')
id_contratos = pd.read_csv('tabelas2/localExecucao.csv')
COLUNA_COMBINADA = 'localExecucao'
NOVAS_COLUNAS = ['pais', 'distrito', 'concelho']

contratos_original[COLUNA_COMBINADA] = contratos_original[COLUNA_COMBINADA].astype(str).str.strip()

contratos_original[NOVAS_COLUNAS] = contratos_original[COLUNA_COMBINADA].str.split(',', n=2, expand=True)
for col in NOVAS_COLUNAS:
    # Tratar valores que possam ter ficado vazios ou NaN
    contratos_original[col] = contratos_original[col].astype(str).str.strip().replace('nan', '') 
contratos_merged = pd.merge(
    contratos_original,
    id_contratos[['pais','distrito','concelho','ID']],
    on=['pais','distrito','concelho'], 
    how='left'         
)
contratos['localExecucao'] = contratos_merged['ID']
contratos['localExecucao'].head()
contratos.to_csv('tabelas2/contratos.csv')

In [36]:
import pandas as pd
contratos = pd.read_csv('tabelas2/contratos.csv')
contratos= contratos.drop(contratos.columns[0],axis=1)
contratos.head()
contratos.to_csv('tabelas2/contratos.csv')


In [37]:
nova_ordem = ['idcontrato','tipoContrato','tipoprocedimento','objectoContrato','adjudicante','adjudicatarios','dataPublicacao','dataCelebracaoContrato','precoContratual','cpv','prazoExecucao','localExecucao','fundamentacao', 'ProcedimentoCentralizado','DescrAcordoQuadro']
contratos = pd.read_csv('tabelas2/contratos.csv')
contratos = contratos[nova_ordem]
contratos.to_csv('tabelas2/contratos.csv')

In [38]:
import pandas as pd
import sqlite3
import os
db_name = 'ContratosPublicos2024.db'

# Estabelece a conexão
# A função 'connect' cria a base de dados se ela não existir
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Lista de ficheiros CSV para importar
csv_files = [
    'contratos.csv',
    'cpv.csv',
    'dataCelebracaoContrato.csv',
    'dataPublicacao.csv',
    'DescrAcordoQuadro.csv',
    'entidades.csv',
    'fundamentacao.csv',
    'idcontrato.csv',
    'localExecucao.csv',
    'objectoContrato.csv',
    'prazoExecucao.csv',
    'precoContratual.csv',
    'ProcedimentoCentralizado.csv',
    'tipoContrato.csv',
    'tipoprocedimento.csv'
]

# Itera sobre a lista e importa cada CSV para uma tabela SQL
for file_name in csv_files:
    try:
        # 1. Lê o ficheiro CSV
        df = pd.read_csv(file_name)
        
        # O nome da tabela será o nome do ficheiro sem a extensão '.csv'
        table_name = file_name.replace('.csv', '')
        
        # 2. Escreve o DataFrame para uma tabela SQLite
        # 'if_exists='replace'' apaga a tabela se ela existir e recria (útil para testes)
        # 'index=False' impede que o índice do DataFrame seja escrito como uma coluna na tabela
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        
        print(f"Sucesso ao importar '{file_name}' para a tabela '{table_name}'.")
        
    except FileNotFoundError:
        print(f"ERRO: Ficheiro '{file_name}' não encontrado.")
    except Exception as e:
        print(f"ERRO ao processar '{file_name}': {e}")
cursor.execute("PRAGMA foreign_keys = ON;")
conn.commit()

# Fecha a conexão
conn.close()

print(f"\nBase de dados '{db_name}' criada/atualizada e conexão fechada.")

Sucesso ao importar 'contratos.csv' para a tabela 'contratos'.
ERRO: Ficheiro 'cpv.csv' não encontrado.
ERRO: Ficheiro 'dataCelebracaoContrato.csv' não encontrado.
ERRO: Ficheiro 'dataPublicacao.csv' não encontrado.
ERRO: Ficheiro 'DescrAcordoQuadro.csv' não encontrado.
Sucesso ao importar 'entidades.csv' para a tabela 'entidades'.
ERRO: Ficheiro 'fundamentacao.csv' não encontrado.
ERRO: Ficheiro 'idcontrato.csv' não encontrado.
ERRO: Ficheiro 'localExecucao.csv' não encontrado.
ERRO: Ficheiro 'objectoContrato.csv' não encontrado.
ERRO: Ficheiro 'prazoExecucao.csv' não encontrado.
ERRO: Ficheiro 'precoContratual.csv' não encontrado.
ERRO: Ficheiro 'ProcedimentoCentralizado.csv' não encontrado.
ERRO: Ficheiro 'tipoContrato.csv' não encontrado.
ERRO: Ficheiro 'tipoprocedimento.csv' não encontrado.

Base de dados 'ContratosPublicos2024.db' criada/atualizada e conexão fechada.
